# 🧭 Compass Feature Pipeline — Walkthrough Completo

> **Objetivo**: Reproduzir passo a passo o que o sistema Compass faz internamente, visualizando cada transformação para entender o Business Case do Model Risk.

---

## Mapa do Notebook

| Seção | O que veremos |
|---|---|
| **1. Setup** | Dependências e imports |
| **2. Geração de Dados** | Como os dados de transações são criados |
| **3. Pipeline Batch** | Cálculo exato da média de 30 dias |
| **4. Pipeline Streaming** | Cálculo incremental via EWMA |
| **5. Feature API** | Como os modelos lêem as features |
| **6. O Problema Central** | Visualizando o Training/Serving Skew |
| **7. Monitoramento Cego** | Por que o dashboard não detecta o problema |
| **8. Score de Crédito** | Impacto real no modelo de decisão |

---
## 1. Setup — Imports e Configuração

Primeiro adicionamos o projeto `compass` ao path para podermos importar seus módulos diretamente.

In [2]:
import sys
import os

# Aponta para o projeto compass-feature-pipeline
COMPASS_SRC = os.path.expanduser('~/Documents/compass-feature-pipeline/src')
COMPASS_ROOT = os.path.expanduser('~/Documents/compass-feature-pipeline')
if COMPASS_SRC not in sys.path:
    sys.path.insert(0, COMPASS_SRC)

import pandas as pd
import numpy as np
import sqlite3
from datetime import UTC, datetime, timedelta
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configurações de display
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', 20)

print('✅ Setup OK')
print(f'   Projeto compass em: {COMPASS_SRC}')

✅ Setup OK
   Projeto compass em: /Users/cristianooliveira/Documents/compass-feature-pipeline/src


---
## 2. Geração de Dados Sintéticos

### 2.1 O que é gerado?

O módulo `data_generation.py` cria um histórico sintético de transações bancárias para **300 clientes** ao longo de **180 dias**. Cada cliente recebe um padrão de comportamento:

| Padrão | Frequência | Proporção |
|---|---|---|
| `daily` | Transações quase todo dia (85% chance/dia) | 40% dos clientes |
| `weekly` | Transações a cada 5-10 dias | 35% dos clientes |
| `sparse` | Surtos curtos separados por longos períodos sem atividade | 25% dos clientes |

Os **valores (amounts)** seguem uma distribuição **log-normal** centrada em compras do dia a dia, com uma chance de 5% de uma transação grande (ex: viagem, eletrodoméstico).

In [3]:
# Importa a função de geração do próprio módulo compass
from compass.data_generation import generate_transactions

# Geramos uma amostra menor (20 clientes, 60 dias) para o walkthrough
# O sistema real usa 300 clientes e 180 dias
df_raw = generate_transactions(num_customers=20, history_days=60, seed=42)

print(f'📦 Dados gerados:')
print(f'   Total de transações : {len(df_raw):,}')
print(f'   Clientes únicos     : {df_raw["customer_id"].nunique()}')
print(f'   Período             : {df_raw["event_timestamp"].min()} → {df_raw["event_timestamp"].max()}')
print(f'   Colunas             : {list(df_raw.columns)}')
print()
print('📋 Primeiras 10 linhas do dataset bruto:')
df_raw.head(10)

📦 Dados gerados:
   Total de transações : 703
   Clientes únicos     : 20
   Período             : 2026-06-16T01:49:56+00:00 → 2026-08-14T20:37:43+00:00
   Colunas             : ['customer_id', 'event_timestamp', 'amount', 'merchant_category', 'channel']

📋 Primeiras 10 linhas do dataset bruto:


,customer_id,event_timestamp,amount,merchant_category,channel
0,C0001,2026-06-16T17:13:10+00:00,7.6000,groceries,pix
1,C0001,2026-06-24T18:51:55+00:00,24.1600,utilities,card_not_present
2,C0001,2026-07-02T18:45:27+00:00,54.1300,transport,card_present
3,C0001,2026-07-09T15:27:09+00:00,67.6600,restaurants,app_transfer
4,C0001,2026-07-16T21:18:33+00:00,11.3200,groceries,card_present
5,C0001,2026-07-25T01:31:53+00:00,54.0900,restaurants,card_not_present
6,C0001,2026-07-30T01:37:48+00:00,20.7700,utilities,card_present
7,C0001,2026-08-07T23:17:48+00:00,21.3500,utilities,app_transfer
8,C0001,2026-08-13T11:56:49+00:00,17.8700,utilities,app_transfer
9,C0002,2026-06-16T01:49:56+00:00,15.4700,groceries,card_not_present


In [4]:
# Visualizando a distribuição de valores por padrão de cliente
print('📊 Estatísticas do campo `amount` (valor da transação):')
print(df_raw['amount'].describe().to_frame().T)
print()

print('📊 Número de transações por cliente (top 5 mais ativos):')
txn_counts = df_raw.groupby('customer_id').size().sort_values(ascending=False)
print(txn_counts.head())
print()
print('📊 Número de transações por cliente (top 5 menos ativos):')
print(txn_counts.tail())

📊 Estatísticas do campo `amount` (valor da transação):
          count    mean     std    min     25%     50%     75%      max
amount 703.0000 41.3160 60.5557 1.7000 13.2800 25.9300 45.9050 646.0500

📊 Número de transações por cliente (top 5 mais ativos):
customer_id
C0019    84
C0015    82
C0005    79
C0010    79
C0020    74
dtype: int64

📊 Número de transações por cliente (top 5 menos ativos):
customer_id
C0004    8
C0003    8
C0009    6
C0018    5
C0011    3
dtype: int64


In [5]:
# Distribuição por canal de pagamento
print('📊 Distribuição por canal:')
print(df_raw['channel'].value_counts().to_frame())
print()
print('📊 Distribuição por categoria:')
print(df_raw['merchant_category'].value_counts().to_frame())

📊 Distribuição por canal:
                  count
channel                
card_present        192
app_transfer        183
pix                 175
card_not_present    153

📊 Distribuição por categoria:
                   count
merchant_category       
entertainment         97
restaurants           95
transport             94
travel                94
utilities             89
healthcare            86
online_retail         83
groceries             65


---
## 3. Pipeline Batch — Média Exata de 30 Dias

### 3.1 O que o Batch Pipeline faz?

O `batch_pipeline.py` roda **uma vez por noite** (como um job agendado). Para cada cliente:

1. Carrega **todo o histórico** de transações do CSV
2. Filtra apenas as transações dos **últimos 30 dias** a partir de uma data de referência (`as_of`)
3. Calcula a **média aritmética exata** do campo `amount` por cliente
4. Persiste o resultado no **Offline Store** (SQLite `offline_store.db`)

**Fórmula:**
$$\text{txn\_amount\_avg\_30d} = \frac{\sum_{i \in \text{janela 30d}} amount_i}{N}$$

### 3.2 Step-by-step visual

In [6]:
# STEP 3.1: Preparação — carregando e normalizando timestamps
df_batch = df_raw.copy()
df_batch['event_timestamp'] = pd.to_datetime(df_batch['event_timestamp'])
if df_batch['event_timestamp'].dt.tz is None:
    df_batch['event_timestamp'] = df_batch['event_timestamp'].dt.tz_localize('UTC')

print('STEP 3.1 — Dataset com timestamps normalizados para UTC:')
print(df_batch[['customer_id', 'event_timestamp', 'amount']].head(8))

STEP 3.1 — Dataset com timestamps normalizados para UTC:
  customer_id           event_timestamp  amount
0       C0001 2026-06-16 17:13:10+00:00  7.6000
1       C0001 2026-06-24 18:51:55+00:00 24.1600
2       C0001 2026-07-02 18:45:27+00:00 54.1300
3       C0001 2026-07-09 15:27:09+00:00 67.6600
4       C0001 2026-07-16 21:18:33+00:00 11.3200
5       C0001 2026-07-25 01:31:53+00:00 54.0900
6       C0001 2026-07-30 01:37:48+00:00 20.7700
7       C0001 2026-08-07 23:17:48+00:00 21.3500


In [7]:
# STEP 3.2: Definindo a data de referência (as_of)
# Normalmente seria datetime.now(UTC), mas usamos o último dia do dataset
AS_OF = df_batch['event_timestamp'].max()
WINDOW_DAYS = 30
WINDOW_START = AS_OF - timedelta(days=WINDOW_DAYS)

print(f'STEP 3.2 — Janela de cálculo:')
print(f'   as_of        : {AS_OF}')
print(f'   window_start : {WINDOW_START}')
print(f'   Duração      : {WINDOW_DAYS} dias')

STEP 3.2 — Janela de cálculo:
   as_of        : 2026-08-14 20:37:43+00:00
   window_start : 2026-07-15 20:37:43+00:00
   Duração      : 30 dias


In [8]:
# STEP 3.3: Filtrando apenas transações dentro da janela de 30 dias
mask = (df_batch['event_timestamp'] >= WINDOW_START) & (df_batch['event_timestamp'] <= AS_OF)
df_windowed = df_batch[mask].copy()

total_antes = len(df_batch)
total_depois = len(df_windowed)

print(f'STEP 3.3 — Filtragem por janela de 30 dias:')
print(f'   Transações antes do filtro : {total_antes:,}')
print(f'   Transações após o filtro   : {total_depois:,}')
print(f'   Transações descartadas     : {total_antes - total_depois:,} ({(total_antes-total_depois)/total_antes*100:.1f}%)')
print()
print('Transações na janela (primeiras 8):')
df_windowed[['customer_id', 'event_timestamp', 'amount']].head(8)

STEP 3.3 — Filtragem por janela de 30 dias:
   Transações antes do filtro : 703
   Transações após o filtro   : 348
   Transações descartadas     : 355 (50.5%)

Transações na janela (primeiras 8):


,customer_id,event_timestamp,amount
4,C0001,2026-07-16 21:18:33+00:00,11.3200
5,C0001,2026-07-25 01:31:53+00:00,54.0900
6,C0001,2026-07-30 01:37:48+00:00,20.7700
7,C0001,2026-08-07 23:17:48+00:00,21.3500
8,C0001,2026-08-13 11:56:49+00:00,17.8700
14,C0002,2026-07-23 19:58:27+00:00,30.2300
15,C0002,2026-07-31 15:16:56+00:00,29.8700
16,C0002,2026-08-06 03:21:14+00:00,581.6500


In [9]:
# STEP 3.4: Calculando a média aritmética EXATA por cliente
batch_averages = df_windowed.groupby('customer_id')['amount'].agg(
    txn_count='count',
    txn_amount_avg_30d='mean',
    txn_total_30d='sum',
    txn_min='min',
    txn_max='max'
).round(4)

print(f'STEP 3.4 — Resultado da agregação BATCH para todos os {len(batch_averages)} clientes:')
batch_averages

STEP 3.4 — Resultado da agregação BATCH para todos os 19 clientes:


,txn_count,txn_amount_avg_30d,txn_total_30d,txn_min,txn_max
customer_id,,,,,
C0001,5,25.0800,125.4000,11.3200,54.0900
C0002,3,213.9167,641.7500,29.8700,581.6500
C0003,4,29.5625,118.2500,8.8500,53.1000
C0004,4,27.3275,109.3100,6.6700,44.6500
C0005,38,37.4558,1423.3200,3.1200,116.6400
C0006,39,27.7454,1082.0700,1.7000,100.3100
C0007,4,40.8925,163.5700,9.6700,77.2500
C0008,4,30.5025,122.0100,2.8000,83.6300
C0009,3,44.1667,132.5000,5.1700,110.0700


In [12]:
# STEP 3.5: Persistindo no Offline Store (SQLite)
# Aqui reproduzimos exatamente o que OfflineStore.write_features() faz

offline_db_path = Path('/tmp/walkthrough_offline.db')
conn_offline = sqlite3.connect(offline_db_path)
conn_offline.execute("""
    CREATE TABLE IF NOT EXISTS offline_features (
        customer_id TEXT NOT NULL,
        as_of TEXT NOT NULL,
        feature_name TEXT NOT NULL,
        value REAL NOT NULL,
        computed_at TEXT NOT NULL,
        PRIMARY KEY (customer_id, as_of, feature_name)
    )
""")
conn_offline.commit()

computed_at = datetime.now(UTC).isoformat()
rows = [
    (customer_id, AS_OF.isoformat(), 'txn_amount_avg_30d', row['txn_amount_avg_30d'], computed_at)
    for customer_id, row in batch_averages.iterrows()
]
conn_offline.executemany("""
    INSERT INTO offline_features (customer_id, as_of, feature_name, value, computed_at)
    VALUES (?, ?, ?, ?, ?)
    ON CONFLICT(customer_id, as_of, feature_name)
    DO UPDATE SET value = excluded.value, computed_at = excluded.computed_at
""", rows)
conn_offline.commit()

# Verificando o que foi salvo
df_offline_store = pd.read_sql('SELECT * FROM offline_features ORDER BY customer_id', conn_offline)
print(f'STEP 3.5 — Offline Store (SQLite) — {len(df_offline_store)} registros gravados:')
print(f'   Chave primária : (customer_id, as_of, feature_name)')
print(f'   Arquivo        : {offline_db_path}')
print()
df_offline_store.head(20)

STEP 3.5 — Offline Store (SQLite) — 19 registros gravados:
   Chave primária : (customer_id, as_of, feature_name)
   Arquivo        : /tmp/walkthrough_offline.db



,customer_id,as_of,feature_name,value,computed_at
0,C0001,2026-08-14T20:37:43+00:00,txn_amount_avg_30d,25.0800,2026-08-15T15:45:37.568810+00:00
1,C0002,2026-08-14T20:37:43+00:00,txn_amount_avg_30d,213.9167,2026-08-15T15:45:37.568810+00:00
2,C0003,2026-08-14T20:37:43+00:00,txn_amount_avg_30d,29.5625,2026-08-15T15:45:37.568810+00:00
3,C0004,2026-08-14T20:37:43+00:00,txn_amount_avg_30d,27.3275,2026-08-15T15:45:37.568810+00:00
4,C0005,2026-08-14T20:37:43+00:00,txn_amount_avg_30d,37.4558,2026-08-15T15:45:37.568810+00:00
5,C0006,2026-08-14T20:37:43+00:00,txn_amount_avg_30d,27.7454,2026-08-15T15:45:37.568810+00:00
6,C0007,2026-08-14T20:37:43+00:00,txn_amount_avg_30d,40.8925,2026-08-15T15:45:37.568810+00:00
7,C0008,2026-08-14T20:37:43+00:00,txn_amount_avg_30d,30.5025,2026-08-15T15:45:37.568810+00:00
8,C0009,2026-08-14T20:37:43+00:00,txn_amount_avg_30d,44.1667,2026-08-15T15:45:37.568810+00:00
9,C0010,2026-08-14T20:37:43+00:00,txn_amount_avg_30d,29.8941,2026-08-15T15:45:37.568810+00:00


---
## 4. Pipeline Streaming — EWMA Incremental

### 4.1 O que o Streaming Pipeline faz?

O `streaming_pipeline.py` roda **continuamente**, processando eventos na ordem em que chegam, **um por um**. Para cada nova transação:

1. Lê o último estado da feature do cliente no **Online Store**
2. Aplica a fórmula EWMA para atualizar o valor
3. Grava o novo valor no **Online Store** (sobrescrevendo o anterior)
4. Salva o índice do último evento processado ("cursor" de posição) para poder retomar de onde parou

**Fórmula EWMA (α = 0.2):**
$$\text{novo\_avg} = \alpha \times \text{amount} + (1 - \alpha) \times \text{avg\_anterior}$$
$$\text{novo\_avg} = 0.2 \times \text{amount} + 0.8 \times \text{avg\_anterior}$$

**Interpretação do α = 0.2:**
- Cada nova transação tem **20% de peso** no cálculo
- O histórico acumulado anterior tem **80% de peso**
- Quanto menor o α, mais "inercial" (lento para mudar) o sistema
- Quanto maior o α, mais o valor acompanha as transações recentes

### 4.2 Step-by-step visual

In [10]:
# STEP 4.1: Ordenando eventos por timestamp (como streaming_pipeline faz)
df_stream = df_raw.copy()
df_stream['event_timestamp'] = pd.to_datetime(df_stream['event_timestamp'])
df_stream.sort_values(['event_timestamp', 'customer_id'], inplace=True, kind='stable')
df_stream.reset_index(drop=True, inplace=True)
df_stream['seq'] = df_stream.index  # cursor sequencial

print('STEP 4.1 — Eventos ordenados cronologicamente com índice sequencial (seq):')
df_stream[['seq', 'customer_id', 'event_timestamp', 'amount']].head(12)

STEP 4.1 — Eventos ordenados cronologicamente com índice sequencial (seq):


,seq,customer_id,event_timestamp,amount
0,0,C0002,2026-06-16 01:49:56+00:00,15.4700
1,1,C0010,2026-06-16 02:41:22+00:00,97.4300
2,2,C0005,2026-06-16 05:31:30+00:00,10.2900
3,3,C0006,2026-06-16 05:52:37+00:00,6.4600
4,4,C0015,2026-06-16 06:33:44+00:00,7.1300
5,5,C0005,2026-06-16 07:26:20+00:00,12.7700
6,6,C0013,2026-06-16 08:55:43+00:00,17.6900
7,7,C0015,2026-06-16 09:20:16+00:00,54.0700
8,8,C0019,2026-06-16 13:58:43+00:00,24.3800
9,9,C0019,2026-06-16 15:19:52+00:00,31.3900


In [11]:
# STEP 4.2: Simulando o EWMA — processando evento a evento para C0001
# Isso revela exatamente como o valor vai sendo construído

ALPHA = 0.2
CUSTOMER = 'C0001'

eventos_c1 = df_stream[df_stream['customer_id'] == CUSTOMER].copy()

ewma_trace = []
running_avg = None

for _, event in eventos_c1.iterrows():
    amount = float(event['amount'])
    if running_avg is None:
        # Primeira transação: sem histórico, inicializa com o próprio valor
        running_avg = amount
        formula = f'Seed: {amount:.2f}'
    else:
        prev = running_avg
        running_avg = ALPHA * amount + (1 - ALPHA) * running_avg
        formula = f'0.2 × {amount:.2f} + 0.8 × {prev:.2f}'
    
    ewma_trace.append({
        'seq': int(event['seq']),
        'event_timestamp': event['event_timestamp'],
        'amount': amount,
        'fórmula EWMA aplicada': formula,
        'txn_amount_avg_30d (EWMA)': round(running_avg, 4)
    })

df_ewma_trace = pd.DataFrame(ewma_trace)
print(f'STEP 4.2 — Trace do EWMA para o cliente {CUSTOMER} (α={ALPHA}):')
print(f'   Total de eventos processados: {len(df_ewma_trace)}')
print(f'   Valor EWMA final            : {df_ewma_trace["txn_amount_avg_30d (EWMA)"].iloc[-1]:.4f}')
print()
df_ewma_trace.head(15)

STEP 4.2 — Trace do EWMA para o cliente C0001 (α=0.2):
   Total de eventos processados: 9
   Valor EWMA final            : 25.6753



,seq,event_timestamp,amount,fórmula EWMA aplicada,txn_amount_avg_30d (EWMA)
0,11,2026-06-16 17:13:10+00:00,7.6000,Seed: 7.60,7.6000
1,100,2026-06-24 18:51:55+00:00,24.1600,0.2 × 24.16 + 0.8 × 7.60,10.9120
2,196,2026-07-02 18:45:27+00:00,54.1300,0.2 × 54.13 + 0.8 × 10.91,19.5556
3,275,2026-07-09 15:27:09+00:00,67.6600,0.2 × 67.66 + 0.8 × 19.56,29.1765
4,366,2026-07-16 21:18:33+00:00,11.3200,0.2 × 11.32 + 0.8 × 29.18,25.6052
5,452,2026-07-25 01:31:53+00:00,54.0900,0.2 × 54.09 + 0.8 × 25.61,31.3021
6,515,2026-07-30 01:37:48+00:00,20.7700,0.2 × 20.77 + 0.8 × 31.30,29.1957
7,622,2026-08-07 23:17:48+00:00,21.3500,0.2 × 21.35 + 0.8 × 29.20,27.6266
8,687,2026-08-13 11:56:49+00:00,17.8700,0.2 × 17.87 + 0.8 × 27.63,25.6753


In [12]:
# STEP 4.3: Rodando o EWMA para todos os clientes e persistindo no Online Store
ALPHA = 0.2

online_db_path = Path('/tmp/walkthrough_online.db')
conn_online = sqlite3.connect(online_db_path)
conn_online.executescript("""
    CREATE TABLE IF NOT EXISTS online_features (
        customer_id TEXT NOT NULL,
        feature_name TEXT NOT NULL,
        value REAL NOT NULL,
        computed_at TEXT NOT NULL,
        PRIMARY KEY (customer_id, feature_name)
    );
    CREATE TABLE IF NOT EXISTS pipeline_state (
        key TEXT PRIMARY KEY,
        value TEXT NOT NULL
    );
""")
conn_online.commit()

running_averages = {}
processed = 0

for _, event in df_stream.iterrows():
    customer_id = event['customer_id']
    amount = float(event['amount'])
    now = datetime.now(UTC).isoformat()
    
    if customer_id in running_averages:
        running_averages[customer_id] = ALPHA * amount + (1 - ALPHA) * running_averages[customer_id]
    else:
        running_averages[customer_id] = amount
    
    conn_online.execute("""
        INSERT INTO online_features (customer_id, feature_name, value, computed_at)
        VALUES (?, 'txn_amount_avg_30d', ?, ?)
        ON CONFLICT(customer_id, feature_name)
        DO UPDATE SET value = excluded.value, computed_at = excluded.computed_at
    """, (customer_id, running_averages[customer_id], now))
    
    conn_online.execute("""
        INSERT INTO pipeline_state (key, value) VALUES ('last_processed_seq', ?)
        ON CONFLICT(key) DO UPDATE SET value = excluded.value
    """, (str(int(event['seq'])),))
    processed += 1

conn_online.commit()

df_online_store = pd.read_sql('SELECT * FROM online_features ORDER BY customer_id', conn_online)
df_pipeline_state = pd.read_sql('SELECT * FROM pipeline_state', conn_online)

print(f'STEP 4.3 — Online Store (SQLite) — {len(df_online_store)} registros (1 por cliente):')
print(f'   Eventos processados : {processed:,}')
print(f'   Cursor final        : {df_pipeline_state["value"].iloc[0]}')
print()
print('Pipeline State (cursor de posição):')
print(df_pipeline_state)
print()
print('Online Features (cada linha é sobrescrita a cada novo evento):')
df_online_store.head(10)

STEP 4.3 — Online Store (SQLite) — 20 registros (1 por cliente):
   Eventos processados : 703
   Cursor final        : 702

Pipeline State (cursor de posição):
                  key value
0  last_processed_seq   702

Online Features (cada linha é sobrescrita a cada novo evento):


,customer_id,feature_name,value,computed_at
0,C0001,txn_amount_avg_30d,25.6753,2026-08-15T15:14:38.445463+00:00
1,C0002,txn_amount_avg_30d,137.3297,2026-08-15T15:14:38.443952+00:00
2,C0003,txn_amount_avg_30d,26.9254,2026-08-15T15:14:38.445073+00:00
3,C0004,txn_amount_avg_30d,29.8571,2026-08-15T15:14:38.444969+00:00
4,C0005,txn_amount_avg_30d,36.5031,2026-08-15T15:14:38.445512+00:00
5,C0006,txn_amount_avg_30d,21.2470,2026-08-15T15:14:38.445658+00:00
6,C0007,txn_amount_avg_30d,31.7588,2026-08-15T15:14:38.445170+00:00
7,C0008,txn_amount_avg_30d,31.6183,2026-08-15T15:14:38.445625+00:00
8,C0009,txn_amount_avg_30d,52.9279,2026-08-15T15:14:38.441938+00:00
9,C0010,txn_amount_avg_30d,27.1059,2026-08-15T15:14:38.445690+00:00


---
## 5. Feature API — Como os Modelos Lêem as Features

A `feature_api.py` é a camada de abstração que esconde os detalhes de armazenamento dos modelos consumidores. Ela oferece duas funções:

| Função | Fonte | Uso |
|---|---|---|
| `get_offline_features(customer_id, as_of)` | `offline_store.db` | Underwriting (decisão de crédito com revisão humana) |
| `get_online_features(customer_id)` | `online_store.db` | In-App (ajuste de limite em tempo real, sem revisão humana) |

In [13]:
# STEP 5.1: Reproduzindo get_offline_features()
# Busca o snapshot mais recente ao ou antes do as_of informado

def get_offline_features(customer_id: str, as_of: datetime, conn) -> dict:
    cursor = conn.execute("""
        SELECT feature_name, value, computed_at
        FROM offline_features
        WHERE customer_id = ? AND as_of = (
            SELECT MAX(as_of) FROM offline_features
            WHERE customer_id = ? AND as_of <= ?
        )
    """, (customer_id, customer_id, as_of.isoformat()))
    return {row[0]: {'value': row[1], 'computed_at': row[2]} for row in cursor.fetchall()}

def get_online_features(customer_id: str, conn) -> dict:
    cursor = conn.execute(
        'SELECT feature_name, value, computed_at FROM online_features WHERE customer_id = ?',
        (customer_id,)
    )
    return {row[0]: {'value': row[1], 'computed_at': row[2]} for row in cursor.fetchall()}

# Testando para C0001
CUSTOMER = 'C0001'
offline = get_offline_features(CUSTOMER, AS_OF, conn_offline)
online = get_online_features(CUSTOMER, conn_online)

print(f'STEP 5.1 — Feature API para cliente {CUSTOMER}:')
print()
print('get_offline_features() → Offline Store:')
print(f'  txn_amount_avg_30d : {offline["txn_amount_avg_30d"]["value"]:.4f}')
print(f'  computed_at        : {offline["txn_amount_avg_30d"]["computed_at"]}')
print()
print('get_online_features() → Online Store:')
print(f'  txn_amount_avg_30d : {online["txn_amount_avg_30d"]["value"]:.4f}')
print(f'  computed_at        : {online["txn_amount_avg_30d"]["computed_at"]}')

STEP 5.1 — Feature API para cliente C0001:

get_offline_features() → Offline Store:
  txn_amount_avg_30d : 25.0800
  computed_at        : 2026-08-15T15:14:38.412556+00:00

get_online_features() → Online Store:
  txn_amount_avg_30d : 25.6753
  computed_at        : 2026-08-15T15:14:38.445463+00:00


---
## 6. ⚠️ O Problema Central — Training/Serving Skew

### 6.1 Comparação direta: Batch vs Streaming

Agora vamos visualizar a divergência para **todos os clientes** e quantificá-la.

In [14]:
# STEP 6.1: Comparando Offline (Batch) vs Online (Streaming) para TODOS os clientes

# Offline: média exata dos 30 dias
df_offline_vals = pd.read_sql(
    'SELECT customer_id, value as offline_exact_mean FROM offline_features',
    conn_offline
)

# Online: último valor EWMA
df_online_vals = pd.read_sql(
    'SELECT customer_id, value as online_ewma FROM online_features',
    conn_online
)

# Merge para comparação
df_comparison = df_offline_vals.merge(df_online_vals, on='customer_id', how='outer')
df_comparison['delta_absoluto'] = (df_comparison['offline_exact_mean'] - df_comparison['online_ewma']).abs()
df_comparison['delta_percentual'] = (
    df_comparison['delta_absoluto'] / df_comparison['offline_exact_mean'] * 100
).round(2)
df_comparison['ewma_mais_alto'] = df_comparison['online_ewma'] > df_comparison['offline_exact_mean']
df_comparison = df_comparison.sort_values('delta_percentual', ascending=False).round(4)

print('STEP 6.1 — Comparação Batch (Treino) vs Streaming (Produção):')
print(f'   Clientes com EWMA MAIOR que média exata : {df_comparison["ewma_mais_alto"].sum()}')
print(f'   Clientes com EWMA MENOR que média exata : {(~df_comparison["ewma_mais_alto"]).sum()}')
print(f'   Delta percentual médio                  : {df_comparison["delta_percentual"].mean():.2f}%')
print(f'   Delta percentual máximo                 : {df_comparison["delta_percentual"].max():.2f}%')
print()
df_comparison

STEP 6.1 — Comparação Batch (Treino) vs Streaming (Produção):
   Clientes com EWMA MAIOR que média exata : 9
   Clientes com EWMA MENOR que média exata : 11
   Delta percentual médio                  : 16.17%
   Delta percentual máximo                 : 62.77%



,customer_id,offline_exact_mean,online_ewma,delta_absoluto,delta_percentual,ewma_mais_alto
17,C0018,34.4200,56.0241,21.6041,62.7700,True
1,C0002,213.9167,137.3297,76.5870,35.8000,False
14,C0015,34.3632,24.8411,9.5221,27.7100,False
5,C0006,27.7454,21.2470,6.4984,23.4200,False
13,C0014,43.1669,52.9308,9.7639,22.6200,True
6,C0007,40.8925,31.7588,9.1337,22.3400,False
8,C0009,44.1667,52.9279,8.7612,19.8400,True
19,C0020,45.3995,36.4488,8.9507,19.7200,False
11,C0012,40.8642,34.2260,6.6382,16.2400,False
15,C0016,20.8950,23.5627,2.6677,12.7700,True


In [15]:
# STEP 6.2: Por que o EWMA diverge da média exata?
# Vamos visualizar isso para o cliente C0001 ao longo do tempo

CUSTOMER = 'C0001'
eventos_c1 = df_stream[df_stream['customer_id'] == CUSTOMER].copy()

# Calcula EWMA acumulado
ewma_vals = []
running = None
for _, row in eventos_c1.iterrows():
    a = float(row['amount'])
    running = a if running is None else ALPHA * a + (1 - ALPHA) * running
    ewma_vals.append(running)
eventos_c1 = eventos_c1.copy()
eventos_c1['ewma'] = ewma_vals

# Calcula média exata rolling 30d em cada ponto
eventos_c1 = eventos_c1.sort_values('event_timestamp')
rolling_means = []
for i, (_, row) in enumerate(eventos_c1.iterrows()):
    ts = row['event_timestamp']
    window = eventos_c1[
        (eventos_c1['event_timestamp'] >= ts - timedelta(days=30)) &
        (eventos_c1['event_timestamp'] <= ts)
    ]
    rolling_means.append(window['amount'].mean())
eventos_c1['exact_30d_mean'] = rolling_means

print(f'STEP 6.2 — Evolução do EWMA vs Média Exata para {CUSTOMER}:')
print(f'   Valor EWMA Final       : {eventos_c1["ewma"].iloc[-1]:.4f}')
print(f'   Valor Média Exata Final: {eventos_c1["exact_30d_mean"].iloc[-1]:.4f}')
print(f'   Diferença              : {abs(eventos_c1["ewma"].iloc[-1] - eventos_c1["exact_30d_mean"].iloc[-1]):.4f}')
print()
eventos_c1[['event_timestamp', 'amount', 'ewma', 'exact_30d_mean']].tail(10)

STEP 6.2 — Evolução do EWMA vs Média Exata para C0001:
   Valor EWMA Final       : 25.6753
   Valor Média Exata Final: 25.0800
   Diferença              : 0.5953



,event_timestamp,amount,ewma,exact_30d_mean
11,2026-06-16 17:13:10+00:00,7.6000,7.6000,7.6000
100,2026-06-24 18:51:55+00:00,24.1600,10.9120,15.8800
196,2026-07-02 18:45:27+00:00,54.1300,19.5556,28.6300
275,2026-07-09 15:27:09+00:00,67.6600,29.1765,38.3875
366,2026-07-16 21:18:33+00:00,11.3200,25.6052,39.3175
452,2026-07-25 01:31:53+00:00,54.0900,31.3021,46.8000
515,2026-07-30 01:37:48+00:00,20.7700,29.1957,41.5940
622,2026-08-07 23:17:48+00:00,21.3500,27.6266,35.0380
687,2026-08-13 11:56:49+00:00,17.8700,25.6753,25.0800


In [16]:
# STEP 6.3: Impacto no Score de Crédito
# Reproduzindo o credit_score.py

INTERCEPT = 1.5
TXN_WEIGHT = -0.01
THRESHOLD = 0.5

def score(txn_amount_avg_30d: float) -> float:
    logit = INTERCEPT + TXN_WEIGHT * txn_amount_avg_30d
    return 1 / (1 + pow(2.718281828, -logit))

def decide(txn_amount_avg_30d: float) -> dict:
    s = score(txn_amount_avg_30d)
    return {'score': round(s, 4), 'approved': s >= THRESHOLD}

# Calculando scores para todos os clientes com batch vs streaming
score_comparison = []
for _, row in df_comparison.iterrows():
    offline_val = row['offline_exact_mean']
    online_val = row['online_ewma']
    
    batch_decision = decide(offline_val)
    stream_decision = decide(online_val)
    
    conflict = batch_decision['approved'] != stream_decision['approved']
    
    score_comparison.append({
        'customer_id': row['customer_id'],
        'offline_feature': round(offline_val, 4),
        'offline_score': batch_decision['score'],
        'offline_approved': batch_decision['approved'],
        'online_feature': round(online_val, 4),
        'online_score': stream_decision['score'],
        'online_approved': stream_decision['approved'],
        '⚠️ CONFLITO': conflict
    })

df_scores = pd.DataFrame(score_comparison)
conflicts = df_scores['⚠️ CONFLITO'].sum()

print(f'STEP 6.3 — Impacto no Score de Crédito:')
print(f'   Total de clientes          : {len(df_scores)}')
print(f'   Decisões conflitantes ⚠️   : {conflicts} ({conflicts/len(df_scores)*100:.1f}%)')
print(f'   (Aprovado no batch mas negado no streaming ou vice-versa)')
print()
df_scores

STEP 6.3 — Impacto no Score de Crédito:
   Total de clientes          : 20
   Decisões conflitantes ⚠️   : 2 (10.0%)
   (Aprovado no batch mas negado no streaming ou vice-versa)



,customer_id,offline_feature,offline_score,offline_approved,online_feature,online_score,online_approved,⚠️ CONFLITO
0,C0018,34.4200,0.7606,True,56.0241,0.7191,True,False
1,C0002,213.9167,0.3454,False,137.3297,0.5316,True,True
2,C0015,34.3632,0.7607,True,24.8411,0.7776,True,False
3,C0006,27.7454,0.7725,True,21.2470,0.7837,True,False
4,C0014,43.1669,0.7443,True,52.9308,0.7253,True,False
5,C0007,40.8925,0.7486,True,31.7588,0.7654,True,False
6,C0009,44.1667,0.7424,True,52.9279,0.7253,True,False
7,C0020,45.3995,0.7400,True,36.4488,0.7569,True,False
8,C0012,40.8642,0.7486,True,34.2260,0.7609,True,False
9,C0016,20.8950,0.7843,True,23.5627,0.7798,True,False


---
## 7. 📊 Monitoramento Cego — Por que o Dashboard Não Detecta

O `dashboard.py` atual mede apenas:
- **Uptime**: % de chamadas que não geraram exceção
- **Latência média**: tempo em ms de cada chamada

Ele **não mede**:
- Diferença entre valor online e offline (Feature Drift)
- Idade do dado no Online Store (Data Freshness)
- Distribuição estatística das features (Value Distribution Shift)

In [17]:
# STEP 7.1: Reproduzindo o dashboard atual (o que ele VÊ)
import time

successes = 0
total = 0
latencies = []

customer_ids = pd.read_sql('SELECT DISTINCT customer_id FROM online_features', conn_online)['customer_id'].tolist()
as_of_now = datetime.now(UTC)

for cid in customer_ids[:10]:  # amostra de 10
    for fn, args in [
        (get_offline_features, (cid, as_of_now, conn_offline)),
        (get_online_features, (cid, conn_online)),
    ]:
        start = time.perf_counter()
        try:
            fn(*args)
            successes += 1
        except:
            pass
        latencies.append((time.perf_counter() - start) * 1000)
        total += 1

uptime = successes / total * 100
avg_latency = sum(latencies) / len(latencies)

print('STEP 7.1 — Dashboard atual (o que o on-call VÊ):')
print(f'  ✅ Requests checked : {total}')
print(f'  ✅ Uptime           : {uptime:.2f}%')
print(f'  ✅ Avg latency      : {avg_latency:.3f} ms')
print()
print('  ⚠️  O dashboard grita TUDO OK!')
print('  ⚠️  Mas vimos na Seção 6.3 que há conflitos de decisão de crédito!')

STEP 7.1 — Dashboard atual (o que o on-call VÊ):
  ✅ Requests checked : 20
  ✅ Uptime           : 100.00%
  ✅ Avg latency      : 0.011 ms

  ⚠️  O dashboard grita TUDO OK!
  ⚠️  Mas vimos na Seção 6.3 que há conflitos de decisão de crédito!


In [18]:
# STEP 7.2: O que um Monitor de QUALIDADE de dados deveria verificar

print('STEP 7.2 — Monitor de Qualidade (o que DEVERIA ser verificado):')
print()

# (A) Frescor dos dados (Staleness)
FRESHNESS_TTL_HOURS = 6
df_freshness = pd.read_sql(
    'SELECT customer_id, feature_name, computed_at FROM online_features',
    conn_online
)
df_freshness['computed_at'] = pd.to_datetime(df_freshness['computed_at'], utc=True)
df_freshness['age_hours'] = (
    pd.Timestamp.now(tz='UTC') - df_freshness['computed_at']
).dt.total_seconds() / 3600
df_freshness['is_stale'] = df_freshness['age_hours'] > FRESHNESS_TTL_HOURS
stale_count = df_freshness['is_stale'].sum()
print(f'  A) Frescor dos dados (TTL = {FRESHNESS_TTL_HOURS}h):')
print(f'     Clientes com dados FRESCOS  : {len(df_freshness) - stale_count}')
print(f'     Clientes com dados CADUCOS  : {stale_count} ← DEVERIA SER 0')
print()

# (B) Feature Drift: diferença relativa offline vs online
DRIFT_THRESHOLD_PCT = 15.0
drifted = df_comparison[df_comparison['delta_percentual'] > DRIFT_THRESHOLD_PCT]
print(f'  B) Feature Drift (threshold = {DRIFT_THRESHOLD_PCT}% de diferença):')
print(f'     Clientes dentro do threshold : {len(df_comparison) - len(drifted)}')
print(f'     Clientes com drift excessivo : {len(drifted)} ← DEVERIA SER 0')
if len(drifted) > 0:
    print()
    print('     Clientes com maior drift:')
    print(drifted[['customer_id', 'offline_exact_mean', 'online_ewma', 'delta_percentual']].head(5).to_string(index=False))
print()

# (C) Decisões conflitantes
print(f'  C) Conflitos de Decisão de Crédito:')
print(f'     Clientes com mesma decisão  : {len(df_scores) - conflicts}')
print(f'     Clientes com decisão oposta : {conflicts} ← RISCO FINANCEIRO DIRETO')

STEP 7.2 — Monitor de Qualidade (o que DEVERIA ser verificado):

  A) Frescor dos dados (TTL = 6h):
     Clientes com dados FRESCOS  : 20
     Clientes com dados CADUCOS  : 0 ← DEVERIA SER 0

  B) Feature Drift (threshold = 15.0% de diferença):
     Clientes dentro do threshold : 11
     Clientes com drift excessivo : 9 ← DEVERIA SER 0

     Clientes com maior drift:
customer_id  offline_exact_mean  online_ewma  delta_percentual
      C0018             34.4200      56.0241           62.7700
      C0002            213.9167     137.3297           35.8000
      C0015             34.3632      24.8411           27.7100
      C0006             27.7454      21.2470           23.4200
      C0014             43.1669      52.9308           22.6200

  C) Conflitos de Decisão de Crédito:
     Clientes com mesma decisão  : 18
     Clientes com decisão oposta : 2 ← RISCO FINANCEIRO DIRETO


---
## 8. 🏦 Score de Crédito — Revisão do Modelo

### 8.1 A fórmula do modelo

O modelo em `credit_score.py` é uma **função logística simples** (sigmoid) que mapeia a feature para uma probabilidade de aprovação:

$$\text{score} = \frac{1}{1 + e^{-(\text{intercept} + w \times \text{txn\_amount\_avg\_30d})}}$$

Onde:
- **Intercept** = 1.5 (viés positivo de base)
- **w** = -0.01 (peso negativo: quanto maior a média de gastos, menor o score)
- **Threshold** = 0.5 (score ≥ 0.5 → aprovado)

### 8.2 Interpretação do risco

In [19]:
# STEP 8.1: Visualizando a curva do modelo de scoring

feature_values = np.linspace(0, 300, 300)
scores = [score(v) for v in feature_values]

# Encontra o ponto de corte (feature onde score = 0.5)
# score = 0.5 quando logit = 0 → 1.5 + (-0.01) * x = 0 → x = 150
cutoff = (0 - 1.5) / (-0.01)

print('STEP 8.1 — Curva do Modelo de Scoring:')
print(f'   Ponto de corte (score = 0.5) : txn_amount_avg_30d = R$ {cutoff:.2f}')
print(f'   Clientes com avg > R${cutoff:.0f} → NEGADOS')
print(f'   Clientes com avg < R${cutoff:.0f} → APROVADOS')
print()

# Tabela de exemplos
example_vals = [50, 100, 142.41, 150, 200, 250]
rows_ex = []
for v in example_vals:
    d = decide(v)
    rows_ex.append({
        'txn_amount_avg_30d': v,
        'logit': round(INTERCEPT + TXN_WEIGHT * v, 4),
        'score': d['score'],
        'decisão': '✅ APROVADO' if d['approved'] else '❌ NEGADO'
    })
pd.DataFrame(rows_ex)

STEP 8.1 — Curva do Modelo de Scoring:
   Ponto de corte (score = 0.5) : txn_amount_avg_30d = R$ 150.00
   Clientes com avg > R$150 → NEGADOS
   Clientes com avg < R$150 → APROVADOS



,txn_amount_avg_30d,logit,score,decisão
0,50.0000,1.0000,0.7311,✅ APROVADO
1,100.0000,0.5000,0.6225,✅ APROVADO
2,142.4100,0.0759,0.5190,✅ APROVADO
3,150.0000,0.0000,0.5000,✅ APROVADO
4,200.0000,-0.5000,0.3775,❌ NEGADO
5,250.0000,-1.0000,0.2689,❌ NEGADO


In [20]:
# STEP 8.2: Demonstrando o risco com um exemplo concreto
print('STEP 8.2 — Cenário de Risco Concreto:')
print()
print('  Imagine o cliente C0001 solicitando um aumento de limite no app:')
print()

# Valores do nosso walkthrough
offline_val = df_comparison[df_comparison['customer_id'] == 'C0001']['offline_exact_mean'].values[0]
online_val = df_comparison[df_comparison['customer_id'] == 'C0001']['online_ewma'].values[0]

offline_dec = decide(offline_val)
online_dec = decide(online_val)

print(f'  [UNDERWRITING - usa Offline Store]')
print(f'    txn_amount_avg_30d = {offline_val:.4f} (média exata 30d)')
print(f'    score              = {offline_dec["score"]}')
print(f'    decisão            = {"✅ APROVADO" if offline_dec["approved"] else "❌ NEGADO"}')
print()
print(f'  [IN-APP AUTO-APROVAÇÃO - usa Online Store]')
print(f'    txn_amount_avg_30d = {online_val:.4f} (EWMA acumulado)')
print(f'    score              = {online_dec["score"]}')
print(f'    decisão            = {"✅ APROVADO" if online_dec["approved"] else "❌ NEGADO"}')
print()

if offline_dec['approved'] != online_dec['approved']:
    print('  ⚠️  CONFLITO DETECTADO: As duas linhas tomam decisões OPOSTAS!')
    print('  ⚠️  Este é o Training/Serving Skew em ação!')
else:
    print(f'  ℹ️  Para este cliente as decisões coincidem, mas o delta de feature é {abs(offline_val-online_val):.4f}')
    print(f'  ℹ️  Com outros clientes este delta causaria decisões opostas (vide Seção 6.3)')

STEP 8.2 — Cenário de Risco Concreto:

  Imagine o cliente C0001 solicitando um aumento de limite no app:

  [UNDERWRITING - usa Offline Store]
    txn_amount_avg_30d = 25.0800 (média exata 30d)
    score              = 0.7772
    decisão            = ✅ APROVADO

  [IN-APP AUTO-APROVAÇÃO - usa Online Store]
    txn_amount_avg_30d = 25.6753 (EWMA acumulado)
    score              = 0.7761
    decisão            = ✅ APROVADO

  ℹ️  Para este cliente as decisões coincidem, mas o delta de feature é 0.5953
  ℹ️  Com outros clientes este delta causaria decisões opostas (vide Seção 6.3)


In [21]:
# STEP 8.3: Resumo Final — Os 3 Riscos do Business Case
print('='*70)
print('STEP 8.3 — RESUMO DOS RISCOS IDENTIFICADOS')
print('='*70)
print()
print('🔴 RISCO 1: Training/Serving Skew')
print('   O modelo foi treinado com Média Exata 30d (Offline Store)')
print('   mas serve em produção com EWMA α=0.2 (Online Store).')
print(f'   → {conflicts} de {len(df_scores)} clientes ({conflicts/len(df_scores)*100:.0f}%) têm decisões conflitantes neste dataset')
print()
print('🟠 RISCO 2: Silent Staleness (Dados Caducos Silenciosos)')
print('   FRESHNESS_TTL = 6h está definido em config.py, mas')
print('   a feature_api.py não verifica a idade do dado ao servir.')
print('   Se o streaming parar, dados com dias de atraso serão servidos')
print('   sem nenhum erro ou alerta.')
print()
print('🟡 RISCO 3: Monitoramento Cego')
print('   O dashboard mede apenas latência e uptime da API.')
print('   Não há monitoramento de:')
print('     - Diferença offline vs online (Feature Drift)')
print('     - Frescor dos dados (Data Freshness / Staleness)')
print('     - Distribuição estatística das features')
print('     - Taxa de conflito de decisões')
print()
print('='*70)
print('CONCLUSÃO: Automatizar aprovações de limite sem resolver estes')
print('3 riscos é inaceitável do ponto de vista de Model Risk (IC5).')
print('='*70)

STEP 8.3 — RESUMO DOS RISCOS IDENTIFICADOS

🔴 RISCO 1: Training/Serving Skew
   O modelo foi treinado com Média Exata 30d (Offline Store)
   mas serve em produção com EWMA α=0.2 (Online Store).
   → 2 de 20 clientes (10%) têm decisões conflitantes neste dataset

🟠 RISCO 2: Silent Staleness (Dados Caducos Silenciosos)
   FRESHNESS_TTL = 6h está definido em config.py, mas
   a feature_api.py não verifica a idade do dado ao servir.
   Se o streaming parar, dados com dias de atraso serão servidos
   sem nenhum erro ou alerta.

🟡 RISCO 3: Monitoramento Cego
   O dashboard mede apenas latência e uptime da API.
   Não há monitoramento de:
     - Diferença offline vs online (Feature Drift)
     - Frescor dos dados (Data Freshness / Staleness)
     - Distribuição estatística das features
     - Taxa de conflito de decisões

CONCLUSÃO: Automatizar aprovações de limite sem resolver estes
3 riscos é inaceitável do ponto de vista de Model Risk (IC5).


In [22]:
# Cleanup: fechando conexões SQLite
conn_offline.close()
conn_online.close()
print('✅ Conexões fechadas. Notebook concluído!')

✅ Conexões fechadas. Notebook concluído!
